# session

> Read and write Claude Code session transcripts

In [ ]:
#| default_exp session

Claude Code stores every conversation as a JSONL transcript, and `claude --resume <session-id>` rebuilds a conversation from one. It does not care who wrote the file. A transcript assembled by hand, including tool calls that never really ran, resumes like any other. So sessions can be mined for data, saved as templates, or built synthetically to give a fresh session worked examples of tool use already in its context. This module finds, reads, and writes them.

In [ ]:
#| export
import json, os, re, uuid
from datetime import datetime, timezone
from fastcore.utils import *

In [ ]:
from fastcore.test import *
from collections import Counter
import tempfile, shutil
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage

## Where sessions live

Each project gets a folder under `~/.claude/projects`, named by the project's absolute path with every character that is not a letter or digit replaced by `-`. The path is resolved first, which matters on macOS, where `/tmp` and `/var` are symlinks into `/private`. The folder for a project in `/tmp/foo` is therefore `-private-tmp-foo`.

In [ ]:
#| export
SESSIONS = Path.home()/'.claude'/'projects'

def sess_dir(
    cwd=None, # Project directory; the current directory if None
):
    "The folder where Claude Code keeps session transcripts for the project at `cwd`"
    return SESSIONS/re.sub(r'[^a-zA-Z0-9]', '-', str(Path(cwd or '.').resolve()))

In [ ]:
sess_dir()

Path('/Users/jhoward/.claude/projects/-Users-jhoward-aai-ws-fastllm-claude-code-nbs')

Underscores are replaced too, which is easy to get wrong when sanitizing by hand:

In [ ]:
test_eq(sess_dir('/a/b_c').name, '-a-b-c')

Claude Code exports the running session's id as `CLAUDE_CODE_SESSION_ID` to every process it spawns, shell commands and MCP servers alike, so a tool can locate the transcript of the very session it is running inside.

In [ ]:
#| export
def cur_sess():
    "The current session id, if running under Claude Code"
    return os.environ.get('CLAUDE_CODE_SESSION_ID')

In [ ]:
cur_sess()

In [ ]:
#| export
def sess_file(
    sid=None, # Session id; the current session if None
    cwd=None, # Project directory; the current directory, then all projects, if None
):
    "Path to the transcript of session `sid` for the project at `cwd`"
    sid = sid or cur_sess()
    p = sess_dir(cwd)/f'{sid}.jsonl'
    if cwd is not None or p.exists(): return p
    return first(SESSIONS.glob(f'*/{sid}.jsonl')) or p

Under Claude Code, `sess_file()` with no arguments is therefore usually the running session's own transcript. Session ids are unique across projects, so when `cwd` is not given and the current directory's folder does not hold the session, `sess_file` looks across all project folders. The defaults then work from anywhere, including a notebook kernel whose working directory is not the project root. One caveat: `claude --resume` gives the resumed session a fresh id in the environment while records keep appending to the original transcript, so the advertised id may have no file (yet).

In [ ]:
test_eq(sess_file('abc', '/a/b_c'), SESSIONS/'-a-b-c'/'abc.jsonl')

## Writing a session

Records are plain dicts, so writing a session comes down to filling the envelope and linking the chain. `mk_rec` fills the envelope for one message. It writes the optional bookkeeping a real transcript carries (`version`, `gitBranch`, `userType`, `permissionMode`, and API metadata on assistant records), not only the six required fields: what Claude Code's LLM side makes of a sparse-but-valid record is close to untestable, so we err towards realistic.

Two records with the same content get different files by default, since ids and timestamps are fresh each call. Sometimes the opposite is wanted: the same history should produce byte-identical records, so the same session id maps to the same file however many times it is rebuilt. `canon` gives a canonical JSON rendering to hash, and `stable_uuid` turns any string into a deterministic uuid. `fastllm_claude_code.core` derives its session and record ids this way, and a session template built from a fixed script can too.

In [ ]:
#| export
CC_VERSION = '2.1.206'

def canon(o):
    "Canonical compact JSON for `o`, key-sorted, for stable hashing"
    return json.dumps(o, sort_keys=True, separators=(',', ':'), ensure_ascii=False)

def stable_uuid(s):
    "A uuid deterministically derived from string `s`"
    return str(uuid.uuid5(uuid.NAMESPACE_URL, s))

In [ ]:
test_eq(canon(dict(b=1, a=2)), canon(dict(a=2, b=1)))
test_eq(stable_uuid('x'), stable_uuid('x'))
assert stable_uuid('x') != stable_uuid('y')

In [ ]:
#| export
def _now(): return datetime.now(timezone.utc).strftime(r'%Y-%m-%dT%H:%M:%S.%f')[:-3]+'Z'

def mk_rec(
    role, # 'user' or 'assistant'
    content, # A string, or a list of content blocks
    cwd='.', # Project directory recorded in the envelope
    uid=None, # Record uuid; random if None
    ts=None, # ISO8601 timestamp; the current time if None
    model='claude-sonnet-4-6', # Recorded in assistant API metadata
    **kwargs, # Extra or overriding envelope fields, e.g. `isCompactSummary=True`
):
    "A transcript record for one conversation message, ready for `save_sess`"
    uid = uid or str(uuid.uuid4())
    msg = dict(type='message', role=role, content=content)
    r = dict(type=role, uuid=uid, parentUuid=None, sessionId=None, timestamp=ts or _now(), cwd=str(Path(cwd).resolve()),
        version=CC_VERSION, gitBranch='HEAD', isSidechain=False, userType='external', permissionMode='default', message=msg)
    if role=='assistant':
        tu = isinstance(content, list) and any(isinstance(b, dict) and b.get('type')=='tool_use' for b in content)
        r['requestId'] = 'req_'+stable_uuid(f'{uid}:req').replace('-', '')[:24]
        msg.update(model=model, id='msg_'+stable_uuid(f'{uid}:msg').replace('-', '')[:24],
            stop_reason='tool_use' if tu else 'end_turn', stop_sequence=None, stop_details=None, usage={})
    return dict(r, **kwargs)

In [ ]:
mk_rec('user', 'Hello!')

{'type': 'user',
 'uuid': 'cefd6a28-ffad-42d3-aeff-53cb1563d942',
 'parentUuid': None,
 'sessionId': None,
 'timestamp': '2026-07-10T06:59:59.445Z',
 'cwd': '/Users/jhoward/aai-ws/fastllm-claude-code/nbs',
 'version': '2.1.206',
 'gitBranch': 'HEAD',
 'isSidechain': False,
 'userType': 'external',
 'permissionMode': 'default',
 'message': {'type': 'message', 'role': 'user', 'content': 'Hello!'}}

Assistant records get deterministic API metadata derived from the record id, and `stop_reason` reflects a trailing tool call:

In [ ]:
tu = [dict(type='tool_use', id='toolu_01', name='probe', input={})]
r = mk_rec('assistant', tu, uid=stable_uuid('demo'), ts='2026-01-01T00:00:00.000Z')
test_eq(r['message']['stop_reason'], 'tool_use')
test_eq(r, mk_rec('assistant', tu, uid=stable_uuid('demo'), ts='2026-01-01T00:00:00.000Z'))
assert 'requestId' in r and 'requestId' not in mk_rec('user', 'hi')

`save_sess` assigns a session id, chains each record to the one before, and writes the file where `claude --resume` will look for it. It re-links `parentUuid` unconditionally, so it is for writing linear conversations. To copy a session while keeping its branch structure, write the records yourself.

In [ ]:
#| export
def save_sess(
    recs, # Records in conversation order, e.g. from `mk_rec`
    sid=None, # Session id; a fresh uuid if None
    cwd=None, # Project directory; the current directory if None
):
    "Chain `recs`, write them as session `sid` for the project at `cwd`, and return `sid`"
    sid,prev = sid or str(uuid.uuid4()),None
    for r in recs: r['sessionId'],r['parentUuid'],prev = sid,prev,r['uuid']
    f = sess_file(sid, cwd or '.')
    f.parent.mkdir(parents=True, exist_ok=True)
    f.write_text(''.join(json.dumps(r)+'\n' for r in recs))
    return sid

## A sample session

The smallest useful synthetic history is a tool call that never ran, whose result carries a fact the model could not know any other way. We write it against a scratch project directory.

In [ ]:
proj = Path(tempfile.mkdtemp())
sample = [
    mk_rec('user', 'Measure the flux please.', cwd=proj),
    mk_rec('assistant', [dict(type='tool_use', id='toolu_01', name='flux_meter', input={})], cwd=proj),
    mk_rec('user', [dict(type='tool_result', tool_use_id='toolu_01', content='flux: 41.7 kilofinches')], cwd=proj),
    mk_rec('assistant', [dict(type='text', text='The flux reading is 41.7 kilofinches.')], cwd=proj),
]
sid = save_sess(sample, cwd=proj)
sid

'f6d6657f-3c3b-46a0-a13a-1cc8915c2f73'

## Reading a session

A transcript is one JSON object per line. `load_sess` wraps each in `dict2obj` so fields read as attributes.

In [ ]:
#| export
def load_sess(
    sid=None, # Session id; the current session if None
    cwd=None, # Project directory; the current directory if None
):
    "The records of session `sid`, as an `L` of attribute-access dicts"
    return L(dict2obj(json.loads(l)) for l in sess_file(sid, cwd).read_text().splitlines())

Reading it back gives exactly what we wrote:

In [ ]:
back = load_sess(sid, proj)
test_eq(len(back), 4)
test_eq(back[-1].message.content[0].text, 'The flux reading is 41.7 kilofinches.')

A record carries more than resume strictly needs. Only six fields are required: `type`, `uuid`, `parentUuid`, `sessionId`, `timestamp`, and `message`. The rest is optional bookkeeping. Strip `timestamp` and the session is not even found. `message` is shaped exactly as the Anthropic API shapes messages: a `role`, plus `content` as either a string or a list of content blocks (`text`, `tool_use`, `tool_result`, `thinking`). Assistant records in real transcripts also carry API metadata (`requestId`, `message.id`, `model`, usage), and none of it is needed on resume. In particular, synthetic histories work without `thinking` blocks.

## The parent chain

Resume does not replay the file top to bottom. Reconstruction starts at the last record and walks `parentUuid` links backwards, so a record nothing links to is dropped (the CLI prints a warning). Rewinding a conversation is what creates such records: the abandoned turns stay in the file, off the final chain. `sess_thread` performs the same walk.

In [ ]:
#| export
def sess_thread(
    recs, # Session records, e.g. from `load_sess`
):
    "The records on the active conversation chain, walking `parentUuid` back from the last record"
    byid = {r.uuid:r for r in recs if 'uuid' in r}
    cur,res = recs.filter(lambda r: 'uuid' in r)[-1],[]
    while cur is not None:
        res.append(cur)
        cur = byid.get(cur.get('parentUuid'))
    return L(reversed(res))

On the sample, every record is on the chain:

In [ ]:
test_eq(sess_thread(back).attrgot('uuid'), back.attrgot('uuid'))

Break a link and the walk stops early, mirroring what resume does with unchained records:

In [ ]:
broken = load_sess(sid, proj)
broken[2].parentUuid = None
test_eq(len(sess_thread(broken)), 2)

## Inside a live session

Since Claude Code exports `CLAUDE_CODE_SESSION_ID` to child processes, code running inside a session can read the very transcript it is part of. The cells below do that when a live transcript exists, and fall back to the sample session otherwise (a plain shell, CI, or a resumed session whose advertised id has no file yet).

In [ ]:
recs = load_sess() if sess_file().exists() else load_sess(sid, proj)
len(recs)

4

The conversation itself is the `user` and `assistant` records. The rest is bookkeeping Claude Code adds as it runs: `attachment` for injected context such as skills and file contents, `system` for hook output, and assorted prompt, mode, and snapshot markers. Compaction adds a `user` record flagged `isCompactSummary`, carrying a summary of everything before it.

In [ ]:
Counter(recs.attrgot('type'))

Counter({'user': 2, 'assistant': 2})

Here is one user record in full:

In [ ]:
first(recs, lambda r: r.type=='user' and isinstance(r.get('message',{}).get('content'), str))

<div class="prose" markdown="1">

```python
{ 'cwd': '/private/var/folders/51/b2_szf2945n072c0vj2cyty40000gn/T/tmpwm8r1gdg',
  'gitBranch': 'HEAD',
  'isSidechain': False,
  'message': { 'content': 'Measure the flux please.',
               'role': 'user',
               'type': 'message'},
  'parentUuid': None,
  'permissionMode': 'default',
  'sessionId': 'f6d6657f-3c3b-46a0-a13a-1cc8915c2f73',
  'timestamp': '2026-07-10T06:59:59.447Z',
  'type': 'user',
  'userType': 'external',
  'uuid': '754795a5-469f-44ba-8e61-ab31112b99a1',
  'version': '2.1.206'}
```

</div>

In [ ]:
t = sess_thread(recs)
len(t), len(recs)

(4, 4)

In a live transcript the gap between the two counts is bookkeeping records that carry no `uuid`, plus any abandoned branches; the sample has neither. Consecutive records on the chain link up:

In [ ]:
assert all(b.parentUuid==a.uuid for a,b in zip(t, t[1:]))

The proof that the sample is a working session: resume it and ask about the planted fact. The Claude Agent SDK drives the same CLI, so `resume` there reads the same files. This spends tokens, so it is excluded from automated tests.

In [ ]:
#| eval: false
opts = ClaudeAgentOptions(resume=sid, cwd=str(proj), model='haiku')
async for m in query(prompt='What is the flux reading? Reply with only the value.', options=opts):
    if isinstance(m, ResultMessage): print(m.result)

## Cleanup

Remove the sample from `~/.claude/projects`, along with the scratch project.

In [ ]:
shutil.rmtree(sess_dir(proj))
shutil.rmtree(proj)

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()